# Can we predict `passed_quiz`?
Protocol: `protocol.yaml`. Group holdout and 5-fold GroupKFold CV on `student_key`; choice by mean CV ROC AUC; the holdout is scored once.

Feature timing (columns.csv): `enrol_year`, `school_code`, `prior_score` (placement test) are known before the session. `session_minutes` and `hints_used` are recorded during the session, before the closing quiz result, so they are used. `student_key` is excluded (entity key).

In [ ]:
import json, hashlib, numpy as np, pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.base import clone

assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='5a0b0221ed641db40322e1a95cae32cc67d3525ecb500813f95d6c109b2a7409'
assert hashlib.sha256(open('holdout.csv','rb').read()).hexdigest()=='ff08c7a88fd7e15b17b05a390393550fdf57ea2b6df584f1d0594cc2ec1f00e6'
df=pd.read_csv('data.csv')
hold_idx=np.sort(pd.read_csv('holdout.csv')['row'].values)
assert len(df)==12453 and len(hold_idx)==2465
train_idx=np.setdiff1d(np.arange(len(df)),hold_idx)
FEATURES=['enrol_year','school_code','prior_score','session_minutes','hints_used']
TARGET='passed_quiz'
X,y,g=df[FEATURES],df[TARGET],df['student_key']
Xtr,ytr,gtr=X.iloc[train_idx],y.iloc[train_idx],g.iloc[train_idx]
Xho,yho=X.iloc[hold_idx],y.iloc[hold_idx]
print(len(train_idx),len(hold_idx),ytr.mean(),yho.mean())
print('student overlap:',len(set(gtr)&set(g.iloc[hold_idx])))

## Candidates (named before any scoring)
Preprocessing (scaling) lives inside the pipelines, so it is fit within each CV fold.

In [ ]:
candidates={
 'baseline_prior':DummyClassifier(strategy='prior'),
 'logreg':make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000)),
 'random_forest':RandomForestClassifier(n_estimators=300,min_samples_leaf=20,random_state=0,n_jobs=-1),
 'hist_gb':HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0),
}
list(candidates)

## Cross-validation on training rows (GroupKFold by student)

In [ ]:
cv=GroupKFold(n_splits=5)
folds=[(tr,va) for tr,va in cv.split(Xtr,ytr,gtr)]
cv_scores={}
for name,m in candidates.items():
    s=[]
    for tr,va in folds:
        mm=clone(m).fit(Xtr.iloc[tr],ytr.iloc[tr])
        s.append(roc_auc_score(ytr.iloc[va],mm.predict_proba(Xtr.iloc[va])[:,1]))
    cv_scores[name]=s
for k,v in cv_scores.items(): print(f'{k:15s} mean {np.mean(v):.4f} sd {np.std(v):.4f}')
cv_folds=[[[int(x) for x in train_idx[tr]],[int(x) for x in train_idx[va]]] for tr,va in folds]
chosen=max(cv_scores,key=lambda k:np.mean(cv_scores[k]))
print('chosen:',chosen)

## Refit chosen candidate on all training rows; score holdout once

In [ ]:
final=clone(candidates[chosen]).fit(Xtr,ytr)
ho_auc=roc_auc_score(yho,final.predict_proba(Xho)[:,1])
print(f'Holdout ROC AUC ({chosen}): {ho_auc:.4f}')

In [ ]:
evidence={
 'features':FEATURES,
 'candidates':list(candidates),
 'cv_folds':cv_folds,
 'cv_scores':{k:[float(a) for a in v] for k,v in cv_scores.items()},
 'chosen':chosen,
 'holdout_rows':[int(x) for x in hold_idx],
 'holdout_scores':{chosen:float(ho_auc)},
}
json.dump(evidence,open('evidence.json','w'))